In [1]:
# nvidia-smi --query-gpu=name,memory.total,memory.used,memory.free,utilization.gpu --format=csv,noheader,nounits
# rm -rf outputs

In [2]:
import os
import json
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
os.environ["HF_HUB_CONNECT_TIMEOUT"] = "60"
os.environ["HF_HUB_READ_TIMEOUT"] = "60"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
from unsloth import FastLanguageModel
from transformers import TrainingArguments
from trl import SFTTrainer
from trl.trainer.sft_trainer import DataCollatorForVisionLanguageModeling
from datasets import Dataset
import matplotlib.pyplot as plt

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
# 1. CẤU HÌNH MODEL & LOA (QUAN TRỌNG)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2-VL-7B-Instruct",
    max_seq_length = 2048,  
    load_in_4bit = True, # Giảm bộ nhớ để chạy được trên GPU cỏ
)

# Giảm số visual tokens để tránh OOM và lỗi truncation khi ảnh quá lớn
tokenizer.image_processor.max_pixels = 256 * 28 * 28
tokenizer.image_processor.min_pixels = 128 * 28 * 28

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,               # Tăng lên 64 để học VQA sâu hơn
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 32,      # Tương ứng với r
    lora_dropout = 0,     
    bias = "none",        
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = True,    # BẬT để ổn định khi dùng rank cao
    loftq_config = None,  
)

==((====))==  Unsloth 2026.4.4: Fast Qwen2_Vl patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 21.951 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


Unsloth: Making `model.base_model.model.model.language_model` require gradients


In [3]:
def build_image_path(image_id, image_lookup, images_root="training-images"):
    filename = image_lookup.get(str(image_id), f"{int(image_id):012d}.jpg")
    return f"{images_root}/{filename}"

def load_vqa_split(json_path, images_root="training-images"):
    with open(json_path, "r", encoding="utf-8") as f:
        train_json = json.load(f)

    image_lookup = train_json["images"]
    raw_data = []
    missing_images = 0

    for ann in train_json["annotations"].values():
        image_path = build_image_path(ann["image_id"], image_lookup, images_root)
        if os.path.exists(image_path):
            raw_data.append(
                {
                    "image_id": ann["image_id"],
                    "question": ann["question"],
                    "answer": ann["answer"],
                    "image_path": image_path,
                }
            )
        else:
            missing_images += 1

    print(f"Kept {len(raw_data)} samples, skipped {missing_images} missing images")
    return raw_data
raw_data = load_vqa_split("vlsp2023_train_data.json", "training-images")
dev_raw_data = load_vqa_split("vlsp2023_dev_data.json", "dev-images")

Kept 30833 samples, skipped 0 missing images
Kept 3545 samples, skipped 0 missing images


In [6]:
raw_data[0]

{'image_id': 1624,
 'question': 'biển ghi gì ?',
 'answer': 'ghi đường hồ chí minh',
 'image_path': 'training-images/000000001624.jpg'}

In [5]:
def build_vqa_messages(question, answer):
    return [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question},
            ],
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": answer}],
        },
    ]

def format_vqa_example(example):
    # Hỗ trợ cả sample đơn lẻ và batch
    is_single = isinstance(example["question"], str)

    image_paths = [example["image_path"]] if is_single else example["image_path"]
    questions = [example["question"]] if is_single else example["question"]
    answers = [example["answer"]] if is_single else example["answer"]

    output_messages = []
    output_images = []

    for image_path, question, answer in zip(image_paths, questions, answers):
        output_messages.append(build_vqa_messages(question, answer))
        output_images.append([image_path])

    return {"messages": output_messages, "images": output_images}

formatting_prompts_func = format_vqa_example

In [7]:
format_vqa_example(raw_data[0])

{'messages': [[{'role': 'user',
    'content': [{'type': 'image'}, {'type': 'text', 'text': 'biển ghi gì ?'}]},
   {'role': 'assistant',
    'content': [{'type': 'text', 'text': 'ghi đường hồ chí minh'}]}]],
 'images': [['training-images/000000001624.jpg']]}

In [6]:
train_dataset = Dataset.from_list(raw_data)
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)

dev_dataset = Dataset.from_list(dev_raw_data)
dev_dataset = dev_dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/30833 [00:00<?, ? examples/s]

Map:   0%|          | 0/3545 [00:00<?, ? examples/s]

In [7]:
def create_vqa_trainer(model, train_dataset, eval_dataset=None, tokenizer=None):
    model.config.use_cache = False

    if tokenizer is None:
        tokenizer = globals().get("tokenizer")
    if tokenizer is None:
        raise ValueError("tokenizer is required for SFTTrainer")

    if hasattr(tokenizer, "image_processor") and tokenizer.image_processor is not None:
        tokenizer.image_processor.max_pixels = 256 * 28 * 28
        tokenizer.image_processor.min_pixels = 128 * 28 * 28

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        args=TrainingArguments(
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            warmup_steps=10,
            num_train_epochs = 3,
            learning_rate=1e-5,
            bf16=True,
            fp16=False,
            optim="adamw_8bit",
            logging_steps=1,
            output_dir="outputs-7B",
            save_strategy="steps",
            save_steps = 200,
            save_total_limit=2,
            remove_unused_columns=False,
        ),
    )

    trainer.data_collator = DataCollatorForVisionLanguageModeling(
        processor=tokenizer,
        max_length=1024,
    )

    return trainer

trainer = create_vqa_trainer(model, train_dataset, dev_dataset)

/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [8]:
def train_with_checkpoint(trainer):
    output_dir = trainer.args.output_dir
    checkpoint_dirs = [
        d for d in os.listdir(output_dir)
        if d.startswith("checkpoint-") and os.path.isdir(os.path.join(output_dir, d))
    ] if os.path.exists(output_dir) else []

    if checkpoint_dirs:
        latest_checkpoint = max(
            checkpoint_dirs,
            key=lambda x: int(x.split("-")[-1])
        )
        checkpoint_path = os.path.join(output_dir, latest_checkpoint)
        print(f"Resume từ checkpoint: {checkpoint_path}")
        return trainer.train(resume_from_checkpoint=checkpoint_path)

    print("Không tìm thấy checkpoint, bắt đầu train mới")
    return trainer.train()
train_with_checkpoint(trainer)

Resume từ checkpoint: outputs-7B/checkpoint-3800


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 30,833 | Num Epochs = 3 | Total steps = 11,565
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 80,740,352 of 8,372,115,968 (0.96% trained)


Step,Training Loss
3801,6.822000
3802,6.743900
3803,6.839800
3804,6.757100
3805,6.745700
3806,6.800400
3807,6.853400
3808,6.842800
3809,6.820700
3810,6.761100


TrainOutput(global_step=11564, training_loss=4.548560172082313, metrics={'train_runtime': 29211.3808, 'train_samples_per_second': 3.167, 'train_steps_per_second': 0.396, 'total_flos': 1.2285514377189181e+18, 'train_loss': 4.548560172082313, 'epoch': 3.0})

In [9]:
def save_final_model(model, tokenizer, save_dir):
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)

    print(f"Da luu model va tokenizer vao: {save_dir}")
save_final_model(model, tokenizer, "outputs7B/final_lora")

Da luu model va tokenizer vao: outputs7B/final_lora


In [10]:
trainer.evaluate(eval_dataset=dev_dataset)

Unsloth: Not an error, but Qwen2VLForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


{'eval_loss': 6.77372407913208,
 'eval_runtime': 669.3784,
 'eval_samples_per_second': 5.296,
 'eval_steps_per_second': 0.663,
 'epoch': 3.0}

In [12]:
def plot(trainer):
    log_history = trainer.state.log_history
    eval_steps = []
    eval_losses = []

    for entry in log_history:
        if 'eval_loss' in entry:
            eval_steps.append(entry['step'])
            eval_losses.append(entry['eval_loss'])

    if eval_losses:
        plt.figure(figsize=(12, 6))
        plt.plot(eval_steps, eval_losses, marker='o', linestyle='-', linewidth=2, markersize=6)
        plt.xlabel('Step', fontsize=12)
        plt.ylabel('Eval Loss', fontsize=12)
        plt.title('Evaluation Loss Over Training', fontsize=14)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        print(f"Eval steps: {eval_steps}")
        print(f"Min loss: {min(eval_losses):.4f} at step {eval_steps[eval_losses.index(min(eval_losses))]}")
    else:
        print("Không có eval_loss trong log history")

In [13]:
def build_inference_inputs(image_path, question,loaded_tokenizer):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": question},
        ],
    }]

    text_prompt = loaded_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = loaded_tokenizer(
        images=[image_path],
        text=[text_prompt],
        return_tensors="pt",
    ).to("cuda")

    return inputs

def ask_vqa(image_path, question, loaded_model, loaded_tokenizer, max_new_tokens=48):
    inputs = build_inference_inputs(image_path, question, loaded_tokenizer)

    with torch.inference_mode():
        output = loaded_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )

    generated_ids = output[:, inputs["input_ids"].shape[1]:]
    return loaded_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

In [20]:
image_path = "test-images/000000008774.jpg"
question = "Hãy mô tả thật chi tiết những gì bạn thấy trong bức ảnh này?"
answer = ask_vqa(image_path, question,model,tokenizer)
answer

'trong bức ảnh có những người mặc áo dài đỏ đang nhảy múa'